# Untitled28 PML: validated x-only absorbing layer

The global plane-wave basis is retained.  Complex coordinate stretching is
applied only in x, beginning at `|x| = 1.5`; the physical region
`|x|, |y| <= 1` is unchanged.  A compact incident packet ends at `x = -0.525`,
leaving a 0.5-unit buffer before the left PML.  The y direction uses a larger
periodic box and is validated separately by convergence.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from double_slit_pml.model import (
    PMLSettings, PlaneWaveModel, compact_packet, evolve,
    project_separable_state, reconstruct, sigma_and_derivative,
)
from double_slit_pml.diagnostics import integrate_rectangle, relative_density_error

In [ ]:
pml = PMLSettings(start=1.5, thickness=2.0, order=4, target_reflection=1e-3)
model = PlaneWaveModel(Lx=pml.outer_half_length, Ly=6.0, nx=80, ny=30, pml=pml)

x_plot = np.linspace(-model.Lx, model.Lx, 1200)
sigma, _ = sigma_and_derivative(x_plot, pml)
plt.plot(x_plot, sigma)
plt.axvspan(-1, 1, alpha=.12, label='region of interest')
plt.xlabel('x'); plt.ylabel(r'$\sigma(x)$'); plt.legend();

In [ ]:
psi0 = project_separable_state(model, lambda x: compact_packet(x, k0=30.0))
times = np.linspace(0.0, 0.2, 21)
states = evolve(model, psi0, times)
probability = np.sum(np.abs(states)**2, axis=1)

plt.plot(times, probability)
plt.xlabel('time'); plt.ylabel('total probability');
probability[-1] / probability[0]

In [ ]:
# Density in the physical window.  The PML itself is outside this plot.
x = np.linspace(-1.0, 1.0, 240)
y = np.linspace(-1.0, 1.0, 180)
density = np.abs(reconstruct(model, states[-1], x, y))**2
plt.imshow(density.T, origin='lower', extent=(-1, 1, -1, 1), aspect='auto', cmap='magma')
plt.xlabel('x'); plt.ylabel('y'); plt.colorbar(label=r'$|\psi|^2$');

## Reproducible validation

Run `python scripts/generate_figures.py` from the repository root.  It compares
this calculation with a conservative `Lx = 8` box, evaluates the upstream
reflection monitor, and repeats the physical-window density calculation for
`Ly = 5, 6, 7`.  The machine-readable results are written to
`figures/validation_metrics.json`.